# Advanced 07 — Non-Human Identity Security & Key Management for Agents

**Scenario:** a production claims agent uses cloud APIs, MCP tools, internal services and CI/CD. Replace static secrets with workload identity and short-lived credentials, isolate keys, implement rotation and then respond to a credential compromise.

The labs use local simulations for standards/cloud services where live infrastructure is not available. They focus on protocol and architecture behavior rather than pretending to provision real SPIRE, KMS or AWS resources.


In [ ]:
from datetime import datetime,timedelta,timezone
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import secrets,base64,json,re,uuid,copy
import pandas as pd
NOW=datetime.now(timezone.utc)


## 1 — NHI inventory

In [ ]:
inventory=[{"id":"agent:claims","type":"logical-agent","owner":"claims-ai","risk":"high"},
{"id":"spiffe://prod.example.org/agent/claims","type":"workload","owner":"claims-ai","risk":"high"},
{"id":"oauth:claims-agent","type":"oauth-client","owner":"claims-platform","risk":"high"}]
pd.DataFrame(inventory)


## 2 — Separate logical agent, workload and credential

In [ ]:
binding={"logical_agent":"agent:claims","deployment":"claims-v18",
"workload":"spiffe://prod.example.org/agent/claims","session":"sess-812",
"credential":"token-991"}
binding


## 3 — Ownership validation

In [ ]:
def owned(i):return bool(i.get("owner"))
all(owned(i) for i in inventory)


## 4 — Risk classification

In [ ]:
risk_weight={"low":1,"medium":2,"high":3,"critical":4}
sorted(inventory,key=lambda x:risk_weight[x["risk"]],reverse=True)


## 5 — Static secret anti-pattern

In [ ]:
legacy={"AWS_ACCESS_KEY_ID":"AKIA...","API_KEY":"permanent-secret"}
print("Problem: credentials live with application configuration.")


## 6 — Short-lived token

In [ ]:
def issue(sub,aud,scope,minutes=10):
    return {"jti":str(uuid.uuid4()),"sub":sub,"aud":aud,"scope":scope,
            "iat":NOW,"exp":NOW+timedelta(minutes=minutes)}
token=issue("agent:claims","claims-api",["claim.read"],10)
token


## 7 — Bootstrap trust

In [ ]:
bootstrap={"platform":"kubernetes","namespace":"prod","service_account":"claims-agent",
"node_attested":True}
bootstrap


## 8 — SPIFFE ID

In [ ]:
spiffe_id="spiffe://prod.example.org/agents/claims"
trust_domain=spiffe_id.split("/")[2]
trust_domain


## 9 — X509-SVID model

In [ ]:
x509_svid={"spiffe_id":spiffe_id,"not_before":NOW,
"not_after":NOW+timedelta(minutes=45),"private_key_exported":False}
x509_svid


## 10 — JWT-SVID audience

In [ ]:
jwt_svid={"sub":spiffe_id,"aud":["token-broker"],"exp":NOW+timedelta(minutes=5)}
def audience_ok(t,a):return a in t["aud"]
audience_ok(jwt_svid,"token-broker"),audience_ok(jwt_svid,"payments-api")


## 11 — Workload selectors

In [ ]:
registration={"spiffe_id":spiffe_id,
"selectors":{"k8s:ns":"prod","k8s:sa":"claims-agent"}}
def matches(reg,observed):return all(observed.get(k)==v for k,v in reg["selectors"].items())
matches(registration,{"k8s:ns":"prod","k8s:sa":"claims-agent"})


## 12 — mTLS identity concept

In [ ]:
peer={"certificate_san":spiffe_id,"trust_domain":"prod.example.org","valid":True}
authenticated=peer["valid"] and peer["certificate_san"]==spiffe_id
authenticated


## 13 — OAuth Client Credentials model

In [ ]:
client={"client_id":"claims-agent","auth_method":"private_key_jwt"}
access=issue(client["client_id"],"claims-api",["claim.read"],10)
access


## 14 — private_key_jwt

In [ ]:
key=Ed25519PrivateKey.generate()
assertion={"iss":"claims-agent","sub":"claims-agent","aud":"https://as.example/token",
"iat":int(NOW.timestamp()),"exp":int((NOW+timedelta(minutes=2)).timestamp()),"jti":str(uuid.uuid4())}
payload=json.dumps(assertion,sort_keys=True).encode()
sig=key.sign(payload);key.public_key().verify(sig,payload)
print("client key possession verified")


## 15 — Bearer token theft

In [ ]:
stolen=copy.deepcopy(access)
print("bearer theft usable until:",stolen["exp"])


## 16 — Sender-constrained token model

In [ ]:
bound_key=Ed25519PrivateKey.generate()
pub=bound_key.public_key().public_bytes_raw()
thumbprint=base64.urlsafe_b64encode(__import__("hashlib").sha256(pub).digest()).decode().rstrip("=")
bound_token={**access,"cnf":{"jkt":thumbprint}}
bound_token["cnf"]


## 17 — DPoP-style request proof

In [ ]:
request={"htm":"POST","htu":"https://claims.example/claim/483","iat":int(NOW.timestamp()),"jti":str(uuid.uuid4())}
msg=json.dumps(request,sort_keys=True).encode()
proof=bound_key.sign(msg)
bound_key.public_key().verify(proof,msg)


## 18 — Replay cache

In [ ]:
seen=set()
def accept_proof(req):
    if req["jti"] in seen:return False
    seen.add(req["jti"]);return True
accept_proof(request),accept_proof(request)


## 19 — Token exchange

In [ ]:
def exchange(subject_token,audience,scope):
    # teaching policy: exchanged scope must be subset of original scope
    if not set(scope).issubset(subject_token["scope"]):raise PermissionError("scope escalation")
    return issue(subject_token["sub"],audience,scope,5)
tool_token=exchange(access,"claims-search",["claim.read"])
tool_token


## 20 — Audience validation

In [ ]:
def usable(t,aud,scope):
    return t["aud"]==aud and scope in t["scope"] and NOW<t["exp"]
usable(tool_token,"claims-search","claim.read"),usable(tool_token,"payments","claim.read")


## 21 — Tool-specific credential fan-out

In [ ]:
tool_creds={
"search":exchange(access,"claims-search",["claim.read"]),
"documents":exchange(access,"doc-api",["claim.read"])}
{k:v["aud"] for k,v in tool_creds.items()}


## 22 — Key purpose separation

In [ ]:
keys={"tls":Ed25519PrivateKey.generate(),"token_signing":Ed25519PrivateKey.generate(),
"delegation":Ed25519PrivateKey.generate()}
len({id(v) for v in keys.values()})==3


## 23 — KMS-style signer

In [ ]:
kms_key=Ed25519PrivateKey.generate()
allowed_callers={"spiffe://prod.example.org/agents/claims"}
def kms_sign(caller,purpose,payload):
    if caller not in allowed_callers or purpose!="audit-attestation":raise PermissionError
    return kms_key.sign(payload)
kms_sign(spiffe_id,"audit-attestation",b"decision:allow")[:10]


## 24 — Envelope encryption

In [ ]:
master=AESGCM.generate_key(256);master_aead=AESGCM(master)
dek=AESGCM.generate_key(256)
nonce=secrets.token_bytes(12);wrapped=master_aead.encrypt(nonce,dek,b"dek")
data_aead=AESGCM(dek);dn=secrets.token_bytes(12)
cipher=data_aead.encrypt(dn,b"legacy secret",b"claims-agent")
len(cipher)


## 25 — Secret manager boundary

In [ ]:
secret_store={"legacy/vendor-api":"encrypted/versioned-value"}
model_context={"tool":"vendor.lookup","credential_reference":"managed-by-tool-gateway"}
model_context


## 26 — AWS role / STS model

In [ ]:
aws_session={"role":"ClaimsAgentRole","temporary":True,
"expires":NOW+timedelta(hours=1),"permissions":["s3:GetObject"]}
aws_session


## 27 — IAM Roles Anywhere model

In [ ]:
roles_anywhere={"trust_anchor":"enterprise-ca","certificate_subject":spiffe_id,
"target_role":"ClaimsAgentRole","returns":"temporary AWS credentials"}
roles_anywhere


## 28 — CI/CD OIDC federation

In [ ]:
ci_assertion={"iss":"https://token.actions.example","sub":"repo:org/agent:ref:main",
"aud":"cloud-sts","short_lived":True}
ci_assertion


## 29 — Build/runtime separation

In [ ]:
authorities={"ci":{"deploy"},"runtime":{"claim.read","claim.update"}}
authorities["ci"].isdisjoint(authorities["runtime"])


## 30 — Prompt secret isolation

In [ ]:
tool_request={"tool":"claims.lookup","arguments":{"claim_id":"483"}}
assert "token" not in json.dumps(tool_request).lower()


## 31 — Log redaction

In [ ]:
patterns=[re.compile(r"Bearer\s+\S+",re.I),re.compile(r"AKIA[A-Z0-9]+")]
def redact(s):
    for p in patterns:s=p.sub("[REDACTED]",s)
    return s
redact("Authorization: Bearer eyJ.secret.token")


## 32 — Rotation

In [ ]:
rotation={"old":"key-v1","new":"key-v2","phase":"overlap"}
rotation["phase"]="new-primary"
rotation["phase"]="old-revoked"
rotation


## 33 — Rotation failure

In [ ]:
failures=["new cert not propagated","stale trust bundle","clock skew","old revoked too early"]
pd.DataFrame({"failure":failures,"test_required":True})


## 34 — Revocation

In [ ]:
revoked_tokens={access["jti"]}
def active(t):return t["jti"] not in revoked_tokens and NOW<t["exp"]
active(access)


## 35 — Private key exfiltration

In [ ]:
key_policy={"exportable":False,"operations":["sign"],"backend":"KMS/HSM"}
key_policy


## 36 — Signing-service policy

In [ ]:
def signing_service(caller,document_type,payload):
    if caller!=spiffe_id or document_type!="governance-attestation":raise PermissionError
    return kms_key.sign(payload)


## 37 — Break-glass

In [ ]:
break_glass={"approved_by":["security","service-owner"],"scope":["claim.read"],
"expires":NOW+timedelta(minutes=20),"ticket":"INC-812"}


## 38 — Incident response

In [ ]:
incident_steps=["quarantine workload","revoke tokens","rotate key","remove delegations",
"invalidate sessions","inspect audit","re-attest workload","reissue credentials","monitor"]
pd.DataFrame({"order":range(1,len(incident_steps)+1),"step":incident_steps})


## 39 — Identity graph

In [ ]:
edges=[("agent:claims",spiffe_id),(spiffe_id,"oauth:claims-agent"),
("oauth:claims-agent","ClaimsAgentRole"),("ClaimsAgentRole","claims-bucket")]
pd.DataFrame(edges,columns=["from","to"])


## 40 — Dormant identities

In [ ]:
assets=[{"id":"client-a","last_used_days":1},{"id":"client-b","last_used_days":120}]
[x for x in assets if x["last_used_days"]>90]


## 41 — Overprivilege analysis

In [ ]:
granted={"read","update","delete","admin"}
used={"read","update"}
candidate_reduction=granted-used
candidate_reduction


## 42 — Detection rules

In [ ]:
events=[
{"event":"KMS sign spike","severity":"high"},
{"event":"old key after rotation","severity":"high"},
{"event":"unexpected token audience","severity":"critical"},
{"event":"CI identity requests runtime secret","severity":"critical"}]
pd.DataFrame(events)


## 43 — Adversarial matrix

In [ ]:
attacks=["token theft","key exfiltration","replay","audience confusion","credential fan-out",
"secret in prompt","secret in logs","rotation failure","CI compromise","signing oracle","break-glass abuse","orphan NHI"]
controls=["sender constraint","KMS/HSM","jti/nonce","aud validation","token exchange",
"tool gateway","redaction","tested overlap","authority separation","sign policy","auto expiry","ownership"]
pd.DataFrame({"attack":attacks,"control":controls})


# 44 — Capstone: Compromised Production Claims Agent

Build and reason through this architecture:

```text
Kubernetes workload
      ↓ attestation
SPIFFE ID + short-lived SVID
      ↓
OAuth token broker / STS
      ↓ token exchange
claims-specific access token
      ↓
Tool Gateway
      ↓
Claims API

Sensitive signing:
Agent → policy-controlled signing service → KMS/HSM

Cloud:
workload federation / role → temporary cloud credential
```

Then simulate:

```text
1. attacker steals bearer token
2. audience prevents use at payments API
3. sender constraint prevents replay without key
4. anomaly triggers quarantine
5. active tokens revoked
6. signing permission disabled
7. workload key rotated
8. SVID reissued after re-attestation
9. delegations removed/re-established
10. audit reconstructs blast radius
```

Production requirements:

- no static cloud access keys;
- no raw secrets in prompts;
- short-lived credentials;
- tool-specific audiences;
- scope attenuation;
- key-purpose separation;
- KMS/HSM for high-value signing;
- automated rotation;
- revocation;
- controlled break-glass;
- inventory and ownership;
- compromise detection;
- tested recovery.


# Review questions

1. Why distinguish logical agent and workload identity?
2. Why are static secrets particularly dangerous for agents?
3. What is the bootstrap problem?
4. What is a SPIFFE ID?
5. How do X509-SVID and JWT-SVID differ?
6. What role does workload attestation play?
7. Why is mTLS authentication not authorization?
8. When is Client Credentials appropriate?
9. What security property does private_key_jwt add over a shared secret?
10. What are sender-constrained tokens?
11. What does DPoP try to prevent?
12. Why is audience restriction essential?
13. What can Token Exchange provide?
14. Why use tool-specific credentials?
15. Why separate keys by purpose?
16. What security property does KMS/HSM key isolation provide?
17. Why must the LLM not receive credentials?
18. Why is rotation also an availability problem?
19. What should a break-glass NHI control include?
20. How would you calculate credential compromise blast radius?
